# Longitudinal face-aging training

This notebook is the server entry point for the single SD1.5 editing model. It uses one directly sampled diffusion timestep per observation—never a 1,000-step forward corruption loop. Run it in the existing `deep_learning` Conda environment after installing `python -m pip install -e ".[auxiliary,notebooks]"`. ArcFace requires Python 3.11+.

In [ ]:
from pathlib import Path
import torch

from data import build_face_aging_dataloaders
from src.model import build_face_aging_diffusion_bundle
from src.loss import FaceAgingDiffusionLoss
from src.training import TRAIN_AGGING_MODEL, run_training_pipeline_validation

## 1. Data

Only `DATASET_ROOT` needs to change when moving from the sample to the full server dataset. Identity-disjoint splits are produced by the loader. The serious baseline uses 256×256 images, batch size 4, and four accumulation micro-batches (effective batch size 16 on one GPU). Training injects 20% deterministic self-pairs; validation and test retain only real longitudinal pairs.

In [ ]:
DATASET_ROOT = Path('/server/path/to/longitudinal_faces')
CHECKPOINT_DIR = Path('/server/path/to/checkpoints/face_aging_minsnr5')
MONITOR_IMAGE = Path('/server/path/to/fixed_monitor_face.jpg')
MONITOR_SOURCE_AGE = 26  # real age in MONITOR_IMAGE; required by relative-age conditioning
FGNET_IMAGES_ROOT = Path('/server/path/to/FGNET/images')

loaders, data_metadata = build_face_aging_dataloaders(
    DATASET_ROOT,
    image_size=256,
    batch_size=4,
    num_workers=4,
    train_pair_strategy='random_target',
    eval_pair_strategy='all',
    horizontal_flip_prob=0.2,
    include_zero_delta_pairs=True, # inject source==target only in the train split
    zero_delta_pair_prob=0.20,      # conservative zero-edit anchor
    include_bidirectional_pairs=True, # preserve all base pairs; reverse ordering on the fly
    reverse_pair_prob=0.20,           # 80/20 forward/reverse among non-self observations
    include_kaggle=True,              # optional complementary FG-NET source
    kaggle_path=FGNET_IMAGES_ROOT,
    kaggle_proportion=0.40,           # adds 0.40 x Colombian observations; 1.0 uses every FG-NET pair
    kaggle_reverse_pair_prob=0.50,    # stronger rejuvenation exposure for FG-NET
    pin_memory=True,
    persistent_workers=True,
    train_drop_last=False,  # partial final accumulation is handled exactly
)

## 2. SD1.5 bundle and auxiliary networks

The U-Net receives `[noisy_target_latent; source_latent]` (8 channels) plus AgeConditioner V2: normalized source/target/delta ages, eight Fourier frequencies, a 3→Fourier→256→1280 MLP, and a learnable gate injected into the timestep embedding. LoRA, expanded `conv_in`, and the age conditioner are trainable; VAE, CLIP, ArcFace and MiVOLO remain frozen. The dictionaries below are the experiment switches: keep them explicit so this notebook remains the reference for every activatable branch.

In [ ]:
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
IDENTITY_MODEL_ID = 'py-feat/arcface_r50'
AGE_MODEL_ID = 'iitolstykh/mivolo_v2'

AGE_CONDITIONER_CONFIG = {
    'use_age_delta_conditioning': True,   # False removes explicit numerical conditioning
    'age_conditioning_mode': 'delta_mlp', # compatibility name for the injection path
    'use_age_conditioner_v2': True,       # False reconstructs legacy scalar-delta V1
    'age_conditioning_version': 'v2_fourier',
    'age_delta_scale': 80.0,              # source/target use fixed scale 100
    'age_condition_hidden_dim': 256,
    'age_condition_output_dim': None,     # inferred from SD1.5 U-Net: 1280
    'num_fourier_frequencies': 8,
    'age_condition_use_raw_scalars': True,
    'age_condition_use_gate': True,       # learnable age_scale, initialized at 1
}

bundle = build_face_aging_diffusion_bundle(
    model_id='stable-diffusion-v1-5/stable-diffusion-v1-5',
    adapter_type='lora',
    rank=16,
    alpha=16,
    dropout=0.0,
    **AGE_CONDITIONER_CONFIG,
    device='cuda',
    dtype=MODEL_DTYPE,
    load_auxiliary_models=True,
    identity_model_id=IDENTITY_MODEL_ID,
    age_model_id=AGE_MODEL_ID,
    auxiliary_dtype=torch.float32,  # ArcFace/MiVOLO normalization paths require FP32
    auxiliary_trust_remote_code=True,  # review/pin age_revision for production
    auxiliary_activation_checkpointing=True,
)

LOSS_CONFIG = {
    'diffusion_weight': 1.0,
    'identity_weight': 0.20,
    'age_weight': 0.05,
    'use_relative_age_loss': True,  # False cleanly ablates only this branch
    'relative_age_weight': 0.05,
    'relative_age_loss_type': 'l1', # 'l1' or 'mse'
    'use_preservation_loss': True,  # False cleanly ablates image-space anchoring
    'preservation_weight': 0.10,
    'preservation_loss_type': 'l1', # 'l1' or 'mse'
    'preservation_max_delta': 2,    # preserve source for |target-source| <= 2
    'use_small_delta_weighting': True,
    'small_delta_threshold': 5,
    'small_delta_weight': 2.0,      # per-sample diffusion multiplier
    'source_age_prediction_mode': 'age_estimator',
    'identity_reference': 'target', # 'source', 'target', or 'both'
    'diffusion_loss_type': 'mse',
    'age_loss_type': 'l1',
    'min_snr_gamma': 5.0,            # None restores plain DDPM weighting
    'auxiliary_every_n_steps': 4,
    'auxiliary_sample_fraction': 0.25,
    'auxiliary_max_timestep': 400,
    'clamp_pred_x0': True,
    'check_finite': True,
    'vae_decode_checkpointing': True,
}

loss_fn = FaceAgingDiffusionLoss(
    scheduler=bundle['scheduler_train'],
    vae=bundle['vae'],
    identity_encoder=bundle['identity_encoder'],
    age_estimator=bundle['age_estimator'],
    **LOSS_CONFIG,
)

## 3. Preflight

This CPU-safe audit checks the uniform timestep and conditioning-dropout distributions. GPU/real-model checks remain explicitly `NOT RUN` unless a server smoke callback is supplied.

In [ ]:
preflight = run_training_pipeline_validation(bundle['scheduler_train'])
preflight

## 4. Train with `TRAIN_AGGING_MODEL`

Parameter groups:

- **Budget:** `num_epochs` is used unless `max_train_steps` is supplied; optimizer steps take precedence.
- **Conservative editing LR:** LoRA `3e-5`, `conv_in` `5e-6`, age MLP `1e-4`; the lower convolution LR protects spatial structure while new source channels begin learning.
- **AgeConditioner V2:** source, target and delta scalars plus Fourier features feed a 256-wide MLP. `age_condition_use_gate=True` exposes a trainable `age_scale`; setting the gate to zero recovers the original timestep path exactly.
- **Relative-age control:** absolute and relative MiVOLO terms remain conservative calibration losses. Disable only the relative branch with `use_relative_age_loss=False`; disable numerical conditioning by rebuilding the bundle with `use_age_delta_conditioning=False`.
- **Prompt regularization:** the baseline selects numeric prompts per sample 70% of the time and generic `photo of a person` prompts 30%. This is independent from CFG conditioning dropout and from the older two-forward `double_prompt_prob` experiment.
- **Referenced CFG monitoring:** direct samples guide target text against the source-age prompt with `age_guidance_scale=3`, instead of guiding age text against an empty prompt at scale 7. `text_reference_mode='null'` restores legacy CFG; `'generic'` is the second ablation.
- **Accumulation:** losses are normalized by the actual number of samples, including an incomplete final window.
- **Zero/small-delta anchoring:** 20% train-only self-pairs teach exact no-edit behavior; decoded L1 preservation uses weight 0.10 for `|delta|<=2`; diffusion contributions receive a 2x weight for `|delta|<=5`. Each switch is independently configurable in `LOSS_CONFIG`.
- **Diffusion:** uniform full-range timesteps and Min-SNR 5 retain generation ability while reducing conflicting timestep gradients. Set `min_snr_gamma=None` for the plain DDPM baseline.
- **CFG preparation:** `conditioning_dropout_prob=0.05` creates 5% text-only, 5% both, 5% image-only dropout, and 85% fully conditioned examples. Identity loss excludes image-dropped samples by default.
- **Posterior policy:** source latent uses the deterministic posterior mean; target latent is sampled during production training.
- **Double prompt:** disabled initially. Set `double_prompt_prob=0.25` only as an explicit FADING-style experiment; the two sequential losses use weights 0.5 + 0.5.
- **Precision:** `auto` prefers BF16, falls back to FP16 on CUDA, and uses GradScaler only for FP16. Trainable parameters remain FP32.
- **Validation/checkpoints:** fixed validation timesteps/noise, best model selected from `val/loss_total`, and exact optimizer/scheduler/RNG resume.

In [ ]:
PROMPT_REGULARIZATION = {
    'target_prompt_policy': 'mixed', # 'numeric', 'generic', or 'mixed'
    'generic_prompt_prob': 0.30,
    'numeric_prompt_prob': 0.70,
}

MONITORING_CFG = {
    'monitoring_text_reference_mode': 'source_age', # 'source_age', 'generic', or 'null'
    'monitoring_age_guidance_scale': 3.0,
    'monitoring_text_guidance_scale': 7.0,           # legacy scale used by null/inverse CFG
    'monitoring_image_guidance_scale': 1.5,
}

training_state = TRAIN_AGGING_MODEL(
    bundle=bundle,
    loss_fn=loss_fn,
    train_loader=loaders['train'],
    val_loader=loaders['val'],

    num_epochs=10,
    max_train_steps=None,
    lr_lora=3e-5,
    lr_conv_in=5e-6,
    lr_age_conditioner=1e-4,
    weight_decay=1e-2,
    conv_in_weight_decay=1e-2,
    age_conditioner_weight_decay=1e-2,
    use_age_delta_conditioning=True,
    age_conditioning_mode='delta_mlp',
    use_age_conditioner_v2=True,
    age_conditioning_version='v2_fourier',
    age_delta_scale=80.0,
    use_relative_age_loss=True,
    relative_age_weight=0.05,
    relative_age_loss_type='l1',
    use_preservation_loss=True,
    preservation_weight=0.10,
    preservation_loss_type='l1',
    preservation_max_delta=2,
    use_small_delta_weighting=True,
    small_delta_threshold=5,
    small_delta_weight=2.0,
    use_bidirectional_training=True, # must match include_bidirectional_pairs in the loader
    reverse_pair_prob=0.20,          # must match the loader exactly
    warmup_ratio=0.05,
    min_lr_ratio=0.10,
    grad_accum_steps=4,
    max_grad_norm=1.0,

    timestep_sampling='uniform',
    min_train_timestep=0,
    max_train_timestep=None,  # full scheduler range
    min_snr_gamma=5.0,
    auxiliary_max_timestep=400,
    conditioning_dropout_prob=0.05,
    **PROMPT_REGULARIZATION,
    identity_loss_on_image_dropped_samples=False,
    sample_source_posterior=False,
    sample_target_posterior=True,
    noise_offset=0.0,
    double_prompt_prob=0.0,

    amp_enabled=True,
    amp_dtype='auto',
    gradient_checkpointing=True,
    enable_xformers=True,

    checkpoint_dir=CHECKPOINT_DIR,
    monitor='val/loss_total',
    monitor_mode='min',
    validate_every_epochs=1,
    deterministic_validation=True,
    validation_seed=2026,
    save_epoch_checkpoints=True,
    max_epoch_checkpoints=5,
    sample_every_epochs=1,
    sample_fn=None,  # None activates built-in inference monitoring below
    monitoring_image=MONITOR_IMAGE,
    monitoring_target_age=[30, 35, 40, 50, 65],  # same image and seed at every epoch
    monitoring_source_age=MONITOR_SOURCE_AGE,
    monitoring_use_inverse_diffusion=False,  # conservative direct-edit baseline
    monitoring_num_inference_steps=30,
    monitoring_strength=0.35,
    **MONITORING_CFG,
    monitoring_seed=2026,  # fixed across epochs
    monitoring_compute_diagnostics=True,  # age prediction + ArcFace cosine CSVs

    seed=42,
    deterministic=False,
    device='auto',
)